<h1>Set operations with overlay</h1>

<p>When working with multiple spatial datasets – especially multiple <em>polygon</em> or <em>line</em> datasets – users often wish to create new shapes based on places where those datasets overlap (or don’t overlap). These manipulations are often referred using the language of sets – intersections, unions, and differences. These types of operations are made available in the GeoPandas library through the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> method.</p>

<p>The basic idea is demonstrated by the graphic below but keep in mind that overlays operate at the DataFrame level, not on individual geometries, and the properties from both are retained. In effect, for every shape in the left
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a>, this operation is executed against every other shape in the right
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html" title="geopandas.GeoDataFrame"><code>GeoDataFrame</code></a>:</p>

<img src="https://geopandas.org/en/stable/_images/overlay_operations.png">

<p><strong>Source: QGIS documentation</strong></p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Note to users familiar with the <em>shapely</em> library: <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> can be thought of as offering versions of the standard <em>shapely</em> set operations that deal with the complexities of applying set operations to two <em>GeoSeries</em>. The standard <em>shapely</em> set operations are also available as
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.html" title="geopandas.GeoSeries"><code>GeoSeries</code></a> methods.</p>
</div>

# <h2>The different overlay operations</h2>

<p>First, create some example data:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [1]: </span><span class="kn">from</span><span class="w"> </span><span class="nn">shapely.geometry</span><span class="w"> </span><span class="kn">import</span> <span class="n">Polygon</span>

<span class="gp">In [2]: </span><span class="n">polys1</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">GeoSeries</span><span class="p">([</span><span class="n">Polygon</span><span class="p">([(</span><span class="mi">0</span><span class="p">,</span><span class="mi">0</span><span class="p">),</span> <span class="p">(</span><span class="mi">2</span><span class="p">,</span><span class="mi">0</span><span class="p">),</span> <span class="p">(</span><span class="mi">2</span><span class="p">,</span><span class="mi">2</span><span class="p">),</span> <span class="p">(</span><span class="mi">0</span><span class="p">,</span><span class="mi">2</span><span class="p">)]),</span>
<span class="gp">   ...: </span>                              <span class="n">Polygon</span><span class="p">([(</span><span class="mi">2</span><span class="p">,</span><span class="mi">2</span><span class="p">),</span> <span class="p">(</span><span class="mi">4</span><span class="p">,</span><span class="mi">2</span><span class="p">),</span> <span class="p">(</span><span class="mi">4</span><span class="p">,</span><span class="mi">4</span><span class="p">),</span> <span class="p">(</span><span class="mi">2</span><span class="p">,</span><span class="mi">4</span><span class="p">)])])</span>
<span class="gp">   ...: </span>

<span class="gp">In [3]: </span><span class="n">polys2</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">GeoSeries</span><span class="p">([</span><span class="n">Polygon</span><span class="p">([(</span><span class="mi">1</span><span class="p">,</span><span class="mi">1</span><span class="p">),</span> <span class="p">(</span><span class="mi">3</span><span class="p">,</span><span class="mi">1</span><span class="p">),</span> <span class="p">(</span><span class="mi">3</span><span class="p">,</span><span class="mi">3</span><span class="p">),</span> <span class="p">(</span><span class="mi">1</span><span class="p">,</span><span class="mi">3</span><span class="p">)]),</span>
<span class="gp">   ...: </span>                              <span class="n">Polygon</span><span class="p">([(</span><span class="mi">3</span><span class="p">,</span><span class="mi">3</span><span class="p">),</span> <span class="p">(</span><span class="mi">5</span><span class="p">,</span><span class="mi">3</span><span class="p">),</span> <span class="p">(</span><span class="mi">5</span><span class="p">,</span><span class="mi">5</span><span class="p">),</span> <span class="p">(</span><span class="mi">3</span><span class="p">,</span><span class="mi">5</span><span class="p">)])])</span>
<span class="gp">   ...: </span>

<span class="gp">In [4]: </span><span class="n">df1</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">GeoDataFrame</span><span class="p">({</span><span class="s1">'geometry'</span><span class="p">:</span> <span class="n">polys1</span><span class="p">,</span> <span class="s1">'df1'</span><span class="p">:[</span><span class="mi">1</span><span class="p">,</span><span class="mi">2</span><span class="p">]})</span>

<span class="gp">In [5]: </span><span class="n">df2</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">GeoDataFrame</span><span class="p">({</span><span class="s1">'geometry'</span><span class="p">:</span> <span class="n">polys2</span><span class="p">,</span> <span class="s1">'df2'</span><span class="p">:[</span><span class="mi">1</span><span class="p">,</span><span class="mi">2</span><span class="p">]})</span>
</pre>
</div>
</div>

<p>These two GeoDataFrames have some overlapping areas:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [6]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">color</span><span class="o">=</span><span class="s1">'red'</span><span class="p">);</span>

<span class="gp">In [7]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">color</span><span class="o">=</span><span class="s1">'green'</span><span class="p">,</span> <span class="n">alpha</span><span class="o">=</span><span class="mf">0.5</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example.png"><img src="https://geopandas.org/en/stable/_images/overlay_example.png" style="width: 5in;"></a>

<p>The above example illustrates the different overlay modes.
The <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> method will determine the set of all individual geometries from overlaying the two input GeoDataFrames.
This result covers the area covered by the two input GeoDataFrames, and also preserves all unique regions defined by the combined boundaries of the two GeoDataFrames.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>For historical reasons, the overlay method is also available as a top-level function
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.overlay.html" title="geopandas.overlay"><code>overlay()</code></a>.
It is recommended to use the method as the function may be deprecated in the future.</p>
</div>

<p>When using <code>how='union'</code>, all those possible geometries are returned:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [8]: </span><span class="n">res_union</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'union'</span><span class="p">)</span>

<span class="gp">In [9]: </span><span class="n">res_union</span>
<span class="gh">Out[9]: </span>
<span class="go">   df1  df2                                           geometry</span>
<span class="go">0  1.0  1.0                POLYGON ((2 2, 2 1, 1 1, 1 2, 2 2))</span>
<span class="go">1  2.0  1.0                POLYGON ((2 2, 2 3, 3 3, 3 2, 2 2))</span>
<span class="go">2  2.0  2.0                POLYGON ((4 4, 4 3, 3 3, 3 4, 4 4))</span>
<span class="go">3  1.0  NaN      POLYGON ((2 0, 0 0, 0 2, 1 2, 1 1, 2 1, 2 0))</span>
<span class="go">4  2.0  NaN  MULTIPOLYGON (((3 4, 3 3, 2 3, 2 4, 3 4)), ((4...</span>
<span class="go">5  NaN  1.0  MULTIPOLYGON (((2 3, 2 2, 1 2, 1 3, 2 3)), ((3...</span>
<span class="go">6  NaN  2.0      POLYGON ((3 5, 5 5, 5 3, 4 3, 4 4, 3 4, 3 5))</span>

<span class="gp">In [10]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">res_union</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">alpha</span><span class="o">=</span><span class="mf">0.5</span><span class="p">,</span> <span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">)</span>

<span class="gp">In [11]: </span><span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>

<span class="gp">In [12]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example_union.png"><img src="https://geopandas.org/en/stable/_images/overlay_example_union.png" style="width: 5in;"></a>

<p>The other <code>how</code> operations will return different subsets of those geometries.
With <code>how='intersection'</code>, it returns only those geometries that are contained by both GeoDataFrames:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [13]: </span><span class="n">res_intersection</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'intersection'</span><span class="p">)</span>

<span class="gp">In [14]: </span><span class="n">res_intersection</span>
<span class="gh">Out[14]: </span>
<span class="go">   df1  df2                             geometry</span>
<span class="go">0    1    1  POLYGON ((2 2, 2 1, 1 1, 1 2, 2 2))</span>
<span class="go">1    2    1  POLYGON ((2 2, 2 3, 3 3, 3 2, 2 2))</span>
<span class="go">2    2    2  POLYGON ((4 4, 4 3, 3 3, 3 4, 4 4))</span>

<span class="gp">In [15]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">res_intersection</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">)</span>

<span class="gp">In [16]: </span><span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>

<span class="gp">In [17]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example_intersection.png"><img src="https://geopandas.org/en/stable/_images/overlay_example_intersection.png" style="width: 5in;"></a>

<p><code>how='symmetric_difference'</code> is the opposite of <code>'intersection'</code> and returns the geometries that are only part of one of the GeoDataFrames but not of both:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [18]: </span><span class="n">res_symdiff</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'symmetric_difference'</span><span class="p">)</span>

<span class="gp">In [19]: </span><span class="n">res_symdiff</span>
<span class="gh">Out[19]: </span>
<span class="go">   df1  df2                                           geometry</span>
<span class="go">0  1.0  NaN      POLYGON ((2 0, 0 0, 0 2, 1 2, 1 1, 2 1, 2 0))</span>
<span class="go">1  2.0  NaN  MULTIPOLYGON (((3 4, 3 3, 2 3, 2 4, 3 4)), ((4...</span>
<span class="go">2  NaN  1.0  MULTIPOLYGON (((2 3, 2 2, 1 2, 1 3, 2 3)), ((3...</span>
<span class="go">3  NaN  2.0      POLYGON ((3 5, 5 5, 5 3, 4 3, 4 4, 3 4, 3 5))</span>

<span class="gp">In [20]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">res_symdiff</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">)</span>

<span class="gp">In [21]: </span><span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>

<span class="gp">In [22]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example_symdiff.png"><img src="https://geopandas.org/en/stable/_images/overlay_example_symdiff.png" style="width: 5in;"></a>

<p>To obtain the geometries that are part of <code>df1</code> but are not contained in <code>df2</code>, you can use <code>how='difference'</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [23]: </span><span class="n">res_difference</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'difference'</span><span class="p">)</span>

<span class="gp">In [24]: </span><span class="n">res_difference</span>
<span class="gh">Out[24]: </span>
<span class="go">                                            geometry  df1</span>
<span class="go">0      POLYGON ((2 0, 0 0, 0 2, 1 2, 1 1, 2 1, 2 0))    1</span>
<span class="go">1  MULTIPOLYGON (((3 4, 3 3, 2 3, 2 4, 3 4)), ((4...    2</span>

<span class="gp">In [25]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">res_difference</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">)</span>

<span class="gp">In [26]: </span><span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>

<span class="gp">In [27]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example_difference.png"><img src="https://geopandas.org/en/stable/_images/overlay_example_difference.png" style="width: 5in;"></a>

<p>Finally, with <code>how='identity'</code>, the result consists of the surface of <code>df1</code>, but with the geometries obtained from overlaying <code>df1</code> with <code>df2</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [28]: </span><span class="n">res_identity</span> <span class="o">=</span> <span class="n">df1</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'identity'</span><span class="p">)</span>

<span class="gp">In [29]: </span><span class="n">res_identity</span>
<span class="gh">Out[29]: </span>
<span class="go">   df1  df2                                           geometry</span>
<span class="go">0    1  1.0                POLYGON ((2 2, 2 1, 1 1, 1 2, 2 2))</span>
<span class="go">1    2  1.0                POLYGON ((2 2, 2 3, 3 3, 3 2, 2 2))</span>
<span class="go">2    2  2.0                POLYGON ((4 4, 4 3, 3 3, 3 4, 4 4))</span>
<span class="go">3    1  NaN      POLYGON ((2 0, 0 0, 0 2, 1 2, 1 1, 2 1, 2 0))</span>
<span class="go">4    2  NaN  MULTIPOLYGON (((3 4, 3 3, 2 3, 2 4, 3 4)), ((4...</span>

<span class="gp">In [30]: </span><span class="n">ax</span> <span class="o">=</span> <span class="n">res_identity</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">)</span>

<span class="gp">In [31]: </span><span class="n">df1</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>

<span class="gp">In [32]: </span><span class="n">df2</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">ax</span><span class="o">=</span><span class="n">ax</span><span class="p">,</span> <span class="n">facecolor</span><span class="o">=</span><span class="s1">'none'</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/overlay_example_identity.png"><img src="https://geopandas.org/en/stable/_images/overlay_example_identity.png" style="width: 5in;"></a>

# <h2>Overlay groceries example</h2>

<p>First, load the Chicago community areas and groceries example datasets and select :</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="0"><span></span><span class="gp">In [33]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">geodatasets</span>

<span class="gp">In [34]: </span><span class="n">chicago</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">read_file</span><span class="p">(</span><span class="n">geodatasets</span><span class="o">.</span><span class="n">get_path</span><span class="p">(</span><span class="s2">"geoda.chicago_commpop"</span><span class="p">))</span>

<span class="gp">In [35]: </span><span class="n">groceries</span> <span class="o">=</span> <span class="n">geopandas</span><span class="o">.</span><span class="n">read_file</span><span class="p">(</span><span class="n">geodatasets</span><span class="o">.</span><span class="n">get_path</span><span class="p">(</span><span class="s2">"geoda.groceries"</span><span class="p">))</span>

<span class="go"># Project to crs that uses meters as distance measure</span>
<span class="gp">In [36]: </span><span class="n">chicago</span> <span class="o">=</span> <span class="n">chicago</span><span class="o">.</span><span class="n">to_crs</span><span class="p">(</span><span class="s2">"ESRI:102003"</span><span class="p">)</span>

<span class="gp">In [37]: </span><span class="n">groceries</span> <span class="o">=</span> <span class="n">groceries</span><span class="o">.</span><span class="n">to_crs</span><span class="p">(</span><span class="s2">"ESRI:102003"</span><span class="p">)</span>
</pre>
</div>
</div>

<p>To illustrate the <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> method, consider the following case in which one wishes to identify the “served” portion of each area – defined as areas within 1km of a grocery store – using a <code>GeoDataFrame</code> of community areas and a <code>GeoDataFrame</code> of groceries.</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="0"><span></span><span class="go"># Look at Chicago:</span>
<span class="gp">In [38]: </span><span class="n">chicago</span><span class="o">.</span><span class="n">plot</span><span class="p">();</span>

<span class="go"># Now buffer groceries to find area within 1km.</span>
<span class="go"># Check CRS -- USA Contiguous Albers Equal Area, units of meters.</span>
<span class="gp">In [39]: </span><span class="n">groceries</span><span class="o">.</span><span class="n">crs</span>
<span class="gh">Out[39]: </span>
<span class="go">&lt;Projected CRS: ESRI:102003&gt;</span>
<span class="go">Name: USA_Contiguous_Albers_Equal_Area_Conic</span>
<span class="go">Axis Info [cartesian]:</span>
<span class="go">- E[east]: Easting (metre)</span>
<span class="go">- N[north]: Northing (metre)</span>
<span class="go">Area of Use:</span>
<span class="go">- name: United States (USA) - CONUS onshore - Alabama; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming.</span>
<span class="go">- bounds: (-124.79, 24.41, -66.91, 49.38)</span>
<span class="go">Coordinate Operation:</span>
<span class="go">- name: USA_Contiguous_Albers_Equal_Area_Conic</span>
<span class="go">- method: Albers Equal Area</span>
<span class="go">Datum: North American Datum 1983</span>
<span class="go">- Ellipsoid: GRS 1980</span>
<span class="go">- Prime Meridian: Greenwich</span>

<span class="go"># make 1km buffer</span>
<span class="gp">In [40]: </span><span class="n">groceries</span><span class="p">[</span><span class="s1">'geometry'</span><span class="p">]</span><span class="o">=</span> <span class="n">groceries</span><span class="o">.</span><span class="n">buffer</span><span class="p">(</span><span class="mi">1000</span><span class="p">)</span>

<span class="gp">In [41]: </span><span class="n">groceries</span><span class="o">.</span><span class="n">plot</span><span class="p">();</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/chicago_basic.png"><img src="https://geopandas.org/en/stable/_images/chicago_basic.png" style="width: 5in;">
</a>
<a href="https://geopandas.org/en/stable/_images/groceries_buffers.png"><img src="https://geopandas.org/en/stable/_images/groceries_buffers.png" style="width: 5in;">
</a>

<p>To select only the portion of community areas within 1km of a grocery, specify the <code>how</code> option to be “intersect”, which creates a new set of polygons where these two layers overlap:</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [42]: </span><span class="n">chicago_cores</span> <span class="o">=</span> <span class="n">chicago</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">groceries</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'intersection'</span><span class="p">)</span>

<span class="gp">In [43]: </span><span class="n">chicago_cores</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">alpha</span><span class="o">=</span><span class="mf">0.5</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">,</span> <span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">);</span>
</pre></div>
</div>
<a href="https://geopandas.org/en/stable/_images/chicago_cores.png"><img src="https://geopandas.org/en/stable/_images/chicago_cores.png" style="width: 5in;"></a>

<p>Changing the <code>how</code> option allows for different types of overlay operations. For example, if you were interested in the portions of Chicago <em>far</em> from groceries (the peripheries), you would compute the difference of the two.</p>

<div class="highlight-ipython notranslate"><div class="highlight" style="position: relative;"><span class="copybutton" title="Hide the prompts and output" data-hidden="false" style="cursor: pointer; position: absolute; top: 0px; right: 0px; font-family: monospace; padding-left: 0.2em; padding-right: 0.2em; border-radius: 0px 3px 0px 0px; user-select: none;">&gt;&gt;&gt;</span><pre tabindex="-1"><span></span><span class="gp">In [44]: </span><span class="n">chicago_peripheries</span> <span class="o">=</span> <span class="n">chicago</span><span class="o">.</span><span class="n">overlay</span><span class="p">(</span><span class="n">groceries</span><span class="p">,</span> <span class="n">how</span><span class="o">=</span><span class="s1">'difference'</span><span class="p">)</span>

<span class="gp">In [45]: </span><span class="n">chicago_peripheries</span><span class="o">.</span><span class="n">plot</span><span class="p">(</span><span class="n">alpha</span><span class="o">=</span><span class="mf">0.5</span><span class="p">,</span> <span class="n">edgecolor</span><span class="o">=</span><span class="s1">'k'</span><span class="p">,</span> <span class="n">cmap</span><span class="o">=</span><span class="s1">'tab10'</span><span class="p">);</span>
</pre>
</div>
</div>
<a href="https://geopandas.org/en/stable/_images/chicago_peripheries.png"><img src="https://geopandas.org/en/stable/_images/chicago_peripheries.png" style="width: 5in;"></a>

# <h2>keep_geom_type keyword</h2>

<p>In default settings, <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> returns only geometries of the same geometry type as GeoDataFrame (left one) has, where Polygon and MultiPolygon is considered as a same type (other types likewise).
You can control this behavior using <code>keep_geom_type</code> option, which is set to True by default.
Once set to False, <code>overlay</code> will return all geometry types resulting from selected set-operation.
Different types can result for example from intersection of touching geometries, where two polygons intersects in a line or a point.</p>

# <h2>More examples</h2>

<p>A larger set of examples of the use of <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.overlay.html" title="geopandas.GeoDataFrame.overlay"><code>overlay()</code></a> can be found
<a href="https://nbviewer.jupyter.org/github/geopandas/geopandas/blob/main/doc/source/gallery/overlays.ipynb">here</a></p>